# UNO2D-H10 on Cahn--Hilliard

This notebook adapts the official temporal UNO architecture to the exact
32x32 Cahn--Hilliard benchmark used by AWFNO. It maps a history of
10 field(s) to one next field and then rolls out recursively.
It is **not a windowed-output UNO**.

At test time it receives the first ten true fields; every later state is predicted.
This protocol gives UNO ten exact test fields. It is therefore an official-protocol diagnostic, not an input-matched comparison with AWFNO, which receives only the initial field.

The official seven-block U-shaped Fourier architecture, width 32, factor
3/4, 40-step differentiable autoregressive training, batch size 16, 500
epochs, Adam, and StepLR schedule are retained. Only the spatial block sizes
and Fourier modes are scaled from 64x64 to 32x32. Cahn--Hilliard mass is
enforced by a mean correction after every predicted step, exactly as in the
controlled Cahn--Hilliard baselines.

The fixed 400/30/70 train/validation/test split, sample-size sweep, and
absolute physical L2 metrics are identical to the AWFNO benchmark. The test
set is never used for fitting or checkpoint selection.

Primary sources:

- https://arxiv.org/abs/2204.11127
- https://github.com/ashiq24/UNO/blob/main/ns_uno2d_main.py
- https://github.com/ashiq24/UNO/blob/main/ns_train_2d.py
- https://github.com/ashiq24/UNO/blob/main/navier_stokes_uno2d.py
- https://github.com/ashiq24/UNO/blob/main/integral_operators.py

In [1]:
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


## Configuration

The default run is the full paper-quality sweep. Set `RUN_SAMPLE_SIZES` to
one value only for a diagnostic run. Keep `FORCE_RETRAIN=False` to resume
safely from the latest ten-epoch checkpoint.

In [2]:
from pathlib import Path

CONFIG = {
    "DATA_PATH": "/content/gdrive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt",
    "OUTPUT_ROOT": "/content/gdrive/MyDrive/anus/cahn_hilliard_uno2d_h10_official_protocol_v1",

    # Exact AWFNO split and sample-size sweep.
    "TRAIN_POOL_SIZE": 400,
    "N_VAL": 30,
    "N_TEST": 70,
    "SPLIT_SEED": 2026,
    "SAMPLE_SIZES": [25, 50, 100, 200, 400],
    "RUN_SAMPLE_SIZES": [25, 50, 100, 200, 400],

    # UNO temporal protocol.
    "HISTORY_LENGTH": 10,
    "TRAIN_ROLLOUT_STEPS": 40,
    "TRAIN_WINDOW_STRIDE": 40,

    # Official UNO2D channel structure, scaled only to the 32x32 grid.
    "WIDTH": 32,
    "FACTOR": 0.75,
    "MODES": [11, 7, 3, 3, 3, 7, 11],
    "NORMALIZE_BLOCKS": False,

    # Official UNO optimization settings.
    "EPOCHS": 500,
    "BATCH_SIZE": 16,
    "EVAL_BATCH_SIZE": 16,
    "LR": 1.0e-3,
    "WEIGHT_DECAY": 1.0e-5,
    "SCHEDULER_STEP": 100,
    "SCHEDULER_GAMMA": 0.5,
    "VAL_ROLLOUT_INTERVAL": 10,
    "CHECKPOINT_INTERVAL": 10,

    # Geometry, evaluation, and Cahn--Hilliard invariant.
    "DT": 0.01,
    "USE_TRAPEZOIDAL_TIME_L2": True,
    "INCLUDE_T0_IN_FULL_L2": True,
    "ENFORCE_MASS": True,

    # Reproducibility and execution.
    "BASE_SEED": 341234,
    "FORCE_RETRAIN": False,
    "RUN_EXACT_HASH_OVERLAP_CHECK": True,
    "NUM_WORKERS": 0,
    "PIN_MEMORY": True,
    "INFERENCE_REPEATS": 3,
    "GRAD_CLIP_NORM": None,
    "USE_AMP": False,
    "USE_TORCH_COMPILE": False,

    # Outputs.
    "SAVE_ROLLOUTS": False,
    "SAVE_PLOTS": True,
    "SHOW_PLOTS": False,
}

DATA_PATH = Path(CONFIG["DATA_PATH"])
OUTPUT_ROOT = Path(CONFIG["OUTPUT_ROOT"])
MODEL_DIR = OUTPUT_ROOT / "models"
RESULT_DIR = OUTPUT_ROOT / "results"
PLOT_DIR = OUTPUT_ROOT / "plots"
for directory in (OUTPUT_ROOT, MODEL_DIR, RESULT_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

assert CONFIG["TRAIN_POOL_SIZE"] + CONFIG["N_VAL"] + CONFIG["N_TEST"] == 500
assert max(CONFIG["SAMPLE_SIZES"]) == CONFIG["TRAIN_POOL_SIZE"]
assert CONFIG["HISTORY_LENGTH"] == 10
assert CONFIG["TRAIN_ROLLOUT_STEPS"] == 40
assert CONFIG["WIDTH"] == 32
assert CONFIG["FACTOR"] == 0.75
assert CONFIG["BATCH_SIZE"] == 16
assert CONFIG["EPOCHS"] == 500
assert CONFIG["USE_AMP"] is False

In [3]:
import copy
import gc
import hashlib
import json
import math
import os
import random
import time
import warnings
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.nn.utils import weight_norm
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings(
    "ignore",
    message="`torch.nn.utils.weight_norm` is deprecated",
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(CONFIG["USE_AMP"] and DEVICE.type == "cuda")
AMP_DTYPE = torch.float16

print("Device:", DEVICE)
print("Automatic mixed precision:", AMP_ENABLED)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GiB):",
        torch.cuda.get_device_properties(0).total_memory / 2**30,
    )
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    return value


with open(OUTPUT_ROOT / "config.json", "w") as stream:
    json.dump(json_ready(CONFIG), stream, indent=2)

Device: cuda
Automatic mixed precision: False
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory (GiB): 94.97076416015625


## Load the exact data artifact and reproduce the fixed split

In [4]:
def torch_load_safe(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Edit CONFIG['DATA_PATH'] if needed."
    )

raw_data = torch_load_safe(DATA_PATH, map_location="cpu")
if "X_input" not in raw_data or "Y_output" not in raw_data:
    raise KeyError("Expected dataset keys 'X_input' and 'Y_output'.")

X_input = raw_data["X_input"].float()
Y_output = raw_data["Y_output"].float()
if X_input.ndim != 4 or X_input.shape[-1] != 1:
    raise ValueError(f"Expected X_input=(N,Nx,Ny,1), got {X_input.shape}.")
if Y_output.ndim != 5 or Y_output.shape[-1] != 1:
    raise ValueError(f"Expected Y_output=(N,T,Nx,Ny,1), got {Y_output.shape}.")

X_np = X_input[..., 0].contiguous().numpy().astype(np.float32)
Y_np = (
    Y_output[..., 0]
    .permute(0, 2, 3, 1)
    .contiguous()
    .numpy()
    .astype(np.float32)
)
U_ALL = np.concatenate([X_np[..., None], Y_np], axis=-1).astype(np.float32)
del X_input, Y_output, X_np, Y_np

if "x_grid" in raw_data:
    x_grid = raw_data["x_grid"].float().reshape(-1).numpy().astype(np.float32)
elif "xg" in raw_data:
    x_grid = raw_data["xg"].float().reshape(-1).numpy().astype(np.float32)
else:
    x_grid = np.linspace(0.0, 1.0, U_ALL.shape[1], dtype=np.float32)

if "y_grid" in raw_data:
    y_grid = raw_data["y_grid"].float().reshape(-1).numpy().astype(np.float32)
elif "yg" in raw_data:
    y_grid = raw_data["yg"].float().reshape(-1).numpy().astype(np.float32)
else:
    y_grid = np.linspace(0.0, 1.0, U_ALL.shape[2], dtype=np.float32)

del raw_data
gc.collect()

N_TOTAL, NX, NY, NT = U_ALL.shape
if (NX, NY, NT) != (32, 32, 101):
    raise ValueError(
        f"Expected the saved 32x32x101 artifact, received {U_ALL.shape}."
    )
if N_TOTAL < 500:
    raise ValueError(f"Need at least 500 trajectories, received {N_TOTAL}.")
if not np.all(np.isfinite(U_ALL)):
    raise ValueError("Dataset contains NaN or Inf.")

DX = float(np.mean(np.diff(x_grid.astype(np.float64))))
DY = float(np.mean(np.diff(y_grid.astype(np.float64))))
DT = float(CONFIG["DT"])
t_grid = np.arange(NT, dtype=np.float32) * DT

rng = np.random.default_rng(CONFIG["SPLIT_SEED"])
permutation = rng.permutation(N_TOTAL)
train_idx = permutation[: CONFIG["TRAIN_POOL_SIZE"]]
val_start = CONFIG["TRAIN_POOL_SIZE"]
val_stop = val_start + CONFIG["N_VAL"]
test_stop = val_stop + CONFIG["N_TEST"]
val_idx = permutation[val_start:val_stop]
test_idx = permutation[val_stop:test_stop]

U_TRAIN_POOL = np.ascontiguousarray(U_ALL[train_idx], dtype=np.float32)
U_VAL = np.ascontiguousarray(U_ALL[val_idx], dtype=np.float32)
U_TEST = np.ascontiguousarray(U_ALL[test_idx], dtype=np.float32)
del U_ALL
gc.collect()

split_path = OUTPUT_ROOT / "split_indices.npz"
if split_path.exists():
    saved_split = np.load(split_path)
    for key, current in (
        ("train_idx", train_idx),
        ("val_idx", val_idx),
        ("test_idx", test_idx),
    ):
        if not np.array_equal(saved_split[key], current):
            raise RuntimeError(
                f"Existing split file disagrees for {key}; use a new OUTPUT_ROOT."
            )
else:
    np.savez(
        split_path,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
        split_seed=np.asarray(CONFIG["SPLIT_SEED"]),
    )

print("\nLoaded Cahn--Hilliard benchmark")
print("--------------------------------")
print("data path       :", DATA_PATH)
print("all data shape  :", (N_TOTAL, NX, NY, NT))
print("training pool   :", U_TRAIN_POOL.shape)
print("validation      :", U_VAL.shape)
print("common test     :", U_TEST.shape)
print("x range / dx    :", (float(x_grid.min()), float(x_grid.max())), DX)
print("y range / dy    :", (float(y_grid.min()), float(y_grid.max())), DY)
print("t range / dt    :", (float(t_grid.min()), float(t_grid.max())), DT)
print("train_idx first :", train_idx[:10])
print("val_idx first   :", val_idx[:10])
print("test_idx first  :", test_idx[:10])


def trajectory_hashes(trajectories):
    hashes = set()
    for trajectory in trajectories:
        contiguous = np.ascontiguousarray(trajectory, dtype=np.float32)
        hashes.add(hashlib.sha256(contiguous.view(np.uint8)).hexdigest())
    return hashes


if CONFIG["RUN_EXACT_HASH_OVERLAP_CHECK"]:
    train_hash = trajectory_hashes(U_TRAIN_POOL)
    val_hash = trajectory_hashes(U_VAL)
    test_hash = trajectory_hashes(U_TEST)
    print("\nExact trajectory overlap")
    print("------------------------")
    print("train_pool vs val :", len(train_hash & val_hash))
    print("train_pool vs test:", len(train_hash & test_hash))
    print("val vs test       :", len(val_hash & test_hash))
    assert not (train_hash & val_hash)
    assert not (train_hash & test_hash)
    assert not (val_hash & test_hash)
    del train_hash, val_hash, test_hash


Loaded Cahn--Hilliard benchmark
--------------------------------
data path       : /content/gdrive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt
all data shape  : (520, 32, 32, 101)
training pool   : (400, 32, 32, 101)
validation      : (30, 32, 32, 101)
common test     : (70, 32, 32, 101)
x range / dx    : (-1.0, 1.0) 0.06451612903225806
y range / dy    : (-1.0, 1.0) 0.06451612903225806
t range / dt    : (0.0, 1.0) 0.01
train_idx first : [253 450 274  42 518 167 391 192  84 145]
val_idx first   : [285 136 466 366 375 317 156 186 358 148]
test_idx first  : [110 237 469 121 234 226 109 164  43 102]

Exact trajectory overlap
------------------------
train_pool vs val : 0
train_pool vs test: 0
val vs test       : 0


## Physical error and mass-conservation utilities

In [5]:
def time_weights(nt):
    weights = np.ones(nt, dtype=np.float64)
    if CONFIG["USE_TRAPEZOIDAL_TIME_L2"] and nt > 1:
        weights[0] = 0.5
        weights[-1] = 0.5
    return weights


def l2_per_timestep_np(prediction, truth):
    if prediction.shape != truth.shape:
        raise ValueError(f"Shape mismatch: {prediction.shape} vs {truth.shape}")
    n_samples = prediction.shape[0]
    output = np.empty((n_samples, prediction.shape[-1]), dtype=np.float64)
    for start in range(0, n_samples, 16):
        stop = min(start + 16, n_samples)
        error = (
            prediction[start:stop].astype(np.float64)
            - truth[start:stop].astype(np.float64)
        )
        output[start:stop] = np.sqrt(
            np.sum(error * error, axis=(1, 2)) * DX * DY + 1.0e-30
        )
    return output


def rollout_metrics(prediction, truth):
    l2_t = l2_per_timestep_np(prediction, truth)
    used = l2_t if CONFIG["INCLUDE_T0_IN_FULL_L2"] else l2_t[:, 1:]
    weights = time_weights(used.shape[1])[None, :]
    full = np.sqrt(
        np.sum(weights * used * used, axis=1) * DT + 1.0e-30
    )
    final = l2_t[:, -1]
    return {
        "full_mean": float(np.mean(full)),
        "full_std": float(np.std(full)),
        "full_median": float(np.median(full)),
        "final_mean": float(np.mean(final)),
        "final_std": float(np.std(final)),
        "final_median": float(np.median(final)),
        "mean_t": np.mean(l2_t, axis=0),
        "std_t": np.std(l2_t, axis=0),
        "median_t": np.median(l2_t, axis=0),
        "all_full": full,
        "all_final": final,
    }


def physical_field_l2_torch(prediction, truth):
    return torch.sqrt(
        torch.sum((prediction - truth) ** 2, dim=(-2, -1))
        * DX * DY
        + 1.0e-30
    )


def enforce_mass_torch(prediction, reference):
    if not CONFIG["ENFORCE_MASS"]:
        return prediction
    target_mean = reference.mean(dim=(-2, -1), keepdim=True)
    predicted_mean = prediction.mean(dim=(-2, -1), keepdim=True)
    return prediction + target_mean - predicted_mean


def mass_drift_np(prediction):
    mass = prediction.mean(axis=(1, 2))
    return np.max(np.abs(mass - mass[:, :1]), axis=1)

## Official-style rollout windows

Each training example is initialized from an exact history. The model then
predicts 40 successive fields, feeding each prediction back into the history.
Gradients pass through all 40 predicted steps, matching the official UNO
temporal training protocol.

In [6]:
class RolloutWindowDataset2D(Dataset):
    def __init__(self, trajectories, history_length, forecast_length, stride):
        self.U = np.asarray(trajectories, dtype=np.float32)
        self.history_length = int(history_length)
        self.forecast_length = int(forecast_length)
        maximum_start = (
            self.U.shape[-1] - self.history_length - self.forecast_length
        )
        if maximum_start < 0:
            raise ValueError(self.U.shape)
        starts = list(range(0, maximum_start + 1, int(stride)))
        if starts[-1] != maximum_start:
            starts.append(maximum_start)
        self.starts = starts

    def __len__(self):
        return self.U.shape[0] * len(self.starts)

    def __getitem__(self, index):
        trajectory = index // len(self.starts)
        start = self.starts[index % len(self.starts)]
        middle = start + self.history_length
        end = middle + self.forecast_length
        history = torch.from_numpy(
            self.U[trajectory, :, :, start:middle]
        )
        future = torch.from_numpy(
            self.U[trajectory, :, :, middle:end]
        )
        return history, future


def relative_l2_sum(prediction, truth):
    batch = prediction.shape[0]
    difference = torch.linalg.vector_norm(
        (prediction - truth).reshape(batch, -1), dim=1
    )
    reference = torch.linalg.vector_norm(
        truth.reshape(batch, -1), dim=1
    )
    return torch.sum(difference / (reference + 1.0e-12))

## UNO2D architecture

This is the official UNO2D integral block: truncated 2D Fourier convolution
plus a learned pointwise map resized by bicubic interpolation. The seven
blocks contract 32 -> 24 -> 16 -> 8, operate at the bottleneck, and expand
8 -> 16 -> 24 -> 32 with U-shaped skip connections.

In [7]:
class SpectralConv2dUNO(nn.Module):
    def __init__(
        self, in_channels, out_channels, dim1, dim2, modes1, modes2
    ):
        super().__init__()
        self.in_channels = int(in_channels)
        self.out_channels = int(out_channels)
        self.dim1 = int(dim1)
        self.dim2 = int(dim2)
        self.modes1 = int(modes1)
        self.modes2 = int(modes2)
        scale = (1.0 / (2.0 * self.in_channels)) ** 0.5
        shape = (
            self.in_channels,
            self.out_channels,
            self.modes1,
            self.modes2,
        )
        self.weights1 = nn.Parameter(
            scale * torch.randn(*shape, dtype=torch.cfloat)
        )
        self.weights2 = nn.Parameter(
            scale * torch.randn(*shape, dtype=torch.cfloat)
        )

    @staticmethod
    def complex_multiply(x, weights):
        return torch.einsum("bixy,ioxy->boxy", x, weights)

    def forward(self, x, dim1=None, dim2=None):
        output_dim1 = self.dim1 if dim1 is None else int(dim1)
        output_dim2 = self.dim2 if dim2 is None else int(dim2)
        max_modes1 = min(x.shape[-2] // 2, output_dim1 // 2)
        max_modes2 = min(x.shape[-1] // 2 + 1, output_dim2 // 2 + 1)
        if self.modes1 > max_modes1 or self.modes2 > max_modes2:
            raise ValueError(
                f"Modes {(self.modes1, self.modes2)} exceed the admissible "
                f"limit {(max_modes1, max_modes2)} for "
                f"{tuple(x.shape[-2:])} -> {(output_dim1, output_dim2)}."
            )

        # FFT operations remain in float32/complex64; AMP is disabled.
        x_ft = torch.fft.rfft2(x.float(), norm="forward")
        out_ft = torch.zeros(
            x.shape[0],
            self.out_channels,
            output_dim1,
            output_dim2 // 2 + 1,
            dtype=torch.cfloat,
            device=x.device,
        )
        out_ft[:, :, : self.modes1, : self.modes2] = (
            self.complex_multiply(
                x_ft[:, :, : self.modes1, : self.modes2],
                self.weights1,
            )
        )
        out_ft[:, :, -self.modes1 :, : self.modes2] = (
            self.complex_multiply(
                x_ft[:, :, -self.modes1 :, : self.modes2],
                self.weights2,
            )
        )
        return torch.fft.irfft2(
            out_ft,
            s=(output_dim1, output_dim2),
            norm="forward",
        )


class PointwiseOp2D(nn.Module):
    def __init__(self, in_channels, out_channels, dim1, dim2):
        super().__init__()
        self.conv = nn.Conv2d(int(in_channels), int(out_channels), 1)
        self.dim1 = int(dim1)
        self.dim2 = int(dim2)

    def forward(self, x, dim1=None, dim2=None):
        output_dim1 = self.dim1 if dim1 is None else int(dim1)
        output_dim2 = self.dim2 if dim2 is None else int(dim2)
        x = self.conv(x)
        return F.interpolate(
            x,
            size=(output_dim1, output_dim2),
            mode="bicubic",
            align_corners=True,
            antialias=True,
        )


class OperatorBlock2D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        dim1,
        dim2,
        modes1,
        modes2,
        normalize=False,
        nonlinearity=True,
    ):
        super().__init__()
        self.conv = SpectralConv2dUNO(
            in_channels, out_channels, dim1, dim2, modes1, modes2
        )
        self.w = PointwiseOp2D(
            in_channels, out_channels, dim1, dim2
        )
        self.normalize = bool(normalize)
        self.nonlinearity = bool(nonlinearity)
        if self.normalize:
            self.normalizer = nn.InstanceNorm2d(
                int(out_channels), affine=True
            )

    def forward(self, x, dim1=None, dim2=None):
        x = self.conv(x, dim1, dim2) + self.w(x, dim1, dim2)
        if self.normalize:
            x = self.normalizer(x)
        if self.nonlinearity:
            x = F.gelu(x)
        return x


class UNO2DHistory(nn.Module):
    """History-to-one-state UNO2D used autoregressively."""

    def __init__(self, x_coordinates, y_coordinates, config):
        super().__init__()
        width = int(config["WIDTH"])
        factor = float(config["FACTOR"])
        history = int(config["HISTORY_LENGTH"])
        modes = list(config["MODES"])
        normalize = bool(config["NORMALIZE_BLOCKS"])
        if len(modes) != 7:
            raise ValueError(modes)

        # Official periodic coordinate embedding: sin/cos in x and y.
        x_phase = 2.0 * np.pi * (
            (x_coordinates - x_coordinates[0])
            / (x_coordinates[-1] - x_coordinates[0])
        )
        y_phase = 2.0 * np.pi * (
            (y_coordinates - y_coordinates[0])
            / (y_coordinates[-1] - y_coordinates[0])
        )
        x_phase, y_phase = np.meshgrid(
            x_phase, y_phase, indexing="ij"
        )
        coordinates = np.stack(
            [
                np.sin(x_phase),
                np.sin(y_phase),
                np.cos(x_phase),
                np.cos(y_phase),
            ],
            axis=-1,
        ).astype(np.float32)
        self.register_buffer(
            "coordinates", torch.from_numpy(coordinates).unsqueeze(0)
        )

        self.fc = nn.Linear(history + 4, width // 2)
        self.fc0 = nn.Linear(width // 2, width)

        c0 = width
        c1 = int(2 * factor * width)
        c2 = int(4 * factor * width)
        c3 = int(8 * factor * width)
        self.L0 = OperatorBlock2D(
            c0, c1, 24, 24, modes[0], modes[0], normalize
        )
        self.L1 = OperatorBlock2D(
            c1, c2, 16, 16, modes[1], modes[1], normalize
        )
        self.L2 = OperatorBlock2D(
            c2, c3, 8, 8, modes[2], modes[2], normalize
        )
        self.L3 = OperatorBlock2D(
            c3, c3, 8, 8, modes[3], modes[3], normalize
        )
        self.L4 = OperatorBlock2D(
            c3, c2, 16, 16, modes[4], modes[4], normalize
        )
        self.L5 = OperatorBlock2D(
            2 * c2, c1, 24, 24, modes[5], modes[5], normalize
        )
        self.L6 = OperatorBlock2D(
            2 * c1, width, 32, 32, modes[6], modes[6], normalize
        )
        self.fc1 = nn.Linear(2 * width, 4 * width)
        self.fc2 = nn.Linear(4 * width, 1)

    def forward(self, history):
        if history.ndim != 4:
            raise ValueError(
                f"Expected (batch,x,y,history), received {history.shape}."
            )
        batch, nx, ny, history_length = history.shape
        if (nx, ny) != (32, 32):
            raise ValueError(f"Expected 32x32, received {(nx, ny)}.")
        if history_length != CONFIG["HISTORY_LENGTH"]:
            raise ValueError(
                f"Expected history {CONFIG['HISTORY_LENGTH']}, "
                f"received {history_length}."
            )

        coordinates = self.coordinates.expand(batch, -1, -1, -1)
        x = torch.cat([history, coordinates], dim=-1)
        x_fc = F.gelu(self.fc(x))
        x_fc0 = F.gelu(self.fc0(x_fc)).permute(0, 3, 1, 2)

        x_c0 = self.L0(x_fc0, 24, 24)
        x_c1 = self.L1(x_c0, 16, 16)
        x_c2 = self.L2(x_c1, 8, 8)
        x_c3 = self.L3(x_c2, 8, 8)
        x_c4 = self.L4(x_c3, 16, 16)
        x_c4 = torch.cat([x_c4, x_c1], dim=1)
        x_c5 = self.L5(x_c4, 24, 24)
        x_c5 = torch.cat([x_c5, x_c0], dim=1)
        x_c6 = self.L6(x_c5, 32, 32)
        x_c6 = torch.cat([x_c6, x_fc0], dim=1)
        x_c6 = x_c6.permute(0, 2, 3, 1).contiguous()
        return self.fc2(F.gelu(self.fc1(x_c6))).squeeze(-1)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


def count_real_scalars(model):
    return sum(
        parameter.numel() * (2 if parameter.is_complex() else 1)
        for parameter in model.parameters()
    )


def state_dict_to_cpu(model):
    return {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }


def optimizer_to_device(optimizer, device):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)


def make_scheduler(optimizer):
    return torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=int(CONFIG["SCHEDULER_STEP"]),
        gamma=float(CONFIG["SCHEDULER_GAMMA"]),
    )


probe_model = UNO2DHistory(x_grid, y_grid, CONFIG).to(DEVICE)
print("\nUNO2D-History architecture")
print("--------------------------")
print("history length      :", CONFIG["HISTORY_LENGTH"])
print("parameters          :", f"{count_parameters(probe_model):,}")
print("stored real scalars :", f"{count_real_scalars(probe_model):,}")
print("block modes         :", CONFIG["MODES"])
print("AMP                 : disabled (complex FFT safety)")
with torch.no_grad():
    probe_input = torch.zeros(
        2, NX, NY, CONFIG["HISTORY_LENGTH"], device=DEVICE
    )
    probe_output = probe_model(probe_input)
assert probe_output.shape == (2, NX, NY), probe_output.shape
assert torch.isfinite(probe_output).all()
del probe_model, probe_input, probe_output
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


UNO2D-History architecture
--------------------------
history length      : 10
parameters          : 3,899,089
stored real scalars : 7,696,081
block modes         : [11, 7, 3, 3, 3, 7, 11]
AMP                 : disabled (complex FFT safety)


## Differentiable 40-step training and recursive evaluation

In [8]:
def project_mass(prediction, reference):
    return enforce_mass_torch(prediction, reference)


def train_one_epoch(model, loader, optimizer):
    model.train()
    relative_sum = 0.0
    physical_sum = 0.0
    count = 0

    for history, future in loader:
        rolling_history = history.to(DEVICE, non_blocking=True)
        future = future.to(DEVICE, non_blocking=True)
        batch = rolling_history.shape[0]
        optimizer.zero_grad(set_to_none=True)
        loss = torch.zeros((), dtype=torch.float32, device=DEVICE)
        batch_physical = 0.0

        for step in range(future.shape[-1]):
            truth = future[:, :, :, step]
            prediction = model(rolling_history)
            prediction = project_mass(
                prediction, rolling_history[:, :, :, -1]
            )
            loss = loss + relative_l2_sum(prediction, truth)
            with torch.no_grad():
                batch_physical += float(
                    physical_field_l2_torch(
                        prediction, truth
                    ).sum().item()
                )
            rolling_history = torch.cat(
                [
                    rolling_history[:, :, :, 1:],
                    prediction.unsqueeze(-1),
                ],
                dim=-1,
            )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                "UNO2D produced a non-finite training loss."
            )
        loss.backward()
        if CONFIG["GRAD_CLIP_NORM"] is not None:
            gradient_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), float(CONFIG["GRAD_CLIP_NORM"])
            )
            if not torch.isfinite(gradient_norm):
                raise FloatingPointError(
                    "UNO2D produced non-finite gradients."
                )
        optimizer.step()

        relative_sum += float(loss.detach().item())
        physical_sum += batch_physical
        count += batch * future.shape[-1]

    return relative_sum / count, physical_sum / count


@torch.no_grad()
def evaluate_windows(model, loader):
    model.eval()
    relative_sum = 0.0
    physical_sum = 0.0
    count = 0
    for history, future in loader:
        rolling_history = history.to(DEVICE, non_blocking=True)
        future = future.to(DEVICE, non_blocking=True)
        batch = rolling_history.shape[0]
        for step in range(future.shape[-1]):
            truth = future[:, :, :, step]
            prediction = model(rolling_history)
            prediction = project_mass(
                prediction, rolling_history[:, :, :, -1]
            )
            relative_sum += float(
                relative_l2_sum(prediction, truth).item()
            )
            physical_sum += float(
                physical_field_l2_torch(
                    prediction, truth
                ).sum().item()
            )
            rolling_history = torch.cat(
                [
                    rolling_history[:, :, :, 1:],
                    prediction.unsqueeze(-1),
                ],
                dim=-1,
            )
        count += batch * future.shape[-1]
    return relative_sum / count, physical_sum / count


@torch.no_grad()
def recursive_rollout(model, initial_history, nt):
    model.eval()
    initial_history = np.asarray(initial_history, dtype=np.float32)
    n_samples = initial_history.shape[0]
    history_length = int(CONFIG["HISTORY_LENGTH"])
    prediction = np.empty(
        (n_samples, NX, NY, nt), dtype=np.float32
    )
    prediction[:, :, :, :history_length] = initial_history

    chunk_size = int(CONFIG["EVAL_BATCH_SIZE"])
    for start in range(0, n_samples, chunk_size):
        stop = min(start + chunk_size, n_samples)
        rolling_history = torch.as_tensor(
            initial_history[start:stop],
            dtype=torch.float32,
            device=DEVICE,
        )
        for step in range(history_length, nt):
            next_state = model(rolling_history)
            next_state = project_mass(
                next_state, rolling_history[:, :, :, -1]
            )
            prediction[start:stop, :, :, step] = (
                next_state.cpu().numpy()
            )
            rolling_history = torch.cat(
                [
                    rolling_history[:, :, :, 1:],
                    next_state.unsqueeze(-1),
                ],
                dim=-1,
            )
    return prediction


@torch.no_grad()
def validation_rollout_error(model):
    prediction = recursive_rollout(
        model,
        U_VAL[:, :, :, : CONFIG["HISTORY_LENGTH"]],
        NT,
    )
    if not np.all(np.isfinite(prediction)):
        raise FloatingPointError(
            "UNO2D produced a non-finite validation rollout."
        )
    value = rollout_metrics(prediction, U_VAL)["full_mean"]
    del prediction
    gc.collect()
    return value


def fit_or_load_uno2d(n_train):
    run_seed = CONFIG["BASE_SEED"] + int(n_train)
    run_directory = MODEL_DIR / f"N{n_train}"
    run_directory.mkdir(parents=True, exist_ok=True)
    label = f"uno2d_h{CONFIG['HISTORY_LENGTH']}"
    done_path = run_directory / f"{label}_done.pt"
    best_path = run_directory / f"{label}_best.pt"
    latest_path = run_directory / f"{label}_latest.pt"

    if CONFIG["FORCE_RETRAIN"]:
        for stale_path in (done_path, best_path, latest_path):
            if stale_path.exists():
                stale_path.unlink()

    model = UNO2DHistory(x_grid, y_grid, CONFIG).to(DEVICE)
    expected_parameters = count_parameters(model)
    expected_real_scalars = count_real_scalars(model)

    if done_path.exists() and not CONFIG["FORCE_RETRAIN"]:
        print(f"Loading completed UNO2D checkpoint: {done_path}")
        checkpoint = torch_load_safe(done_path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state"])
        if count_parameters(model) != int(checkpoint["parameters"]):
            raise RuntimeError("Parameter count disagrees with checkpoint.")
        return {
            "model": model,
            "history": checkpoint["history"],
            "best_epoch": int(checkpoint["best_epoch"]),
            "best_val_full": float(checkpoint["best_val_full"]),
            "train_time_sec": float(checkpoint["train_time_sec"]),
            "checkpoint": str(done_path),
        }

    set_seed(run_seed)
    model = UNO2DHistory(x_grid, y_grid, CONFIG).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(CONFIG["LR"]),
        weight_decay=float(CONFIG["WEIGHT_DECAY"]),
    )
    scheduler = make_scheduler(optimizer)

    train_dataset = RolloutWindowDataset2D(
        U_TRAIN_POOL[:n_train],
        CONFIG["HISTORY_LENGTH"],
        CONFIG["TRAIN_ROLLOUT_STEPS"],
        CONFIG["TRAIN_WINDOW_STRIDE"],
    )
    validation_dataset = RolloutWindowDataset2D(
        U_VAL,
        CONFIG["HISTORY_LENGTH"],
        CONFIG["TRAIN_ROLLOUT_STEPS"],
        CONFIG["TRAIN_WINDOW_STRIDE"],
    )
    shuffle_generator = torch.Generator()
    shuffle_generator.manual_seed(run_seed + 100000)
    train_loader = DataLoader(
        train_dataset,
        batch_size=int(CONFIG["BATCH_SIZE"]),
        shuffle=True,
        generator=shuffle_generator,
        num_workers=int(CONFIG["NUM_WORKERS"]),
        pin_memory=bool(
            CONFIG["PIN_MEMORY"] and DEVICE.type == "cuda"
        ),
        drop_last=False,
    )
    validation_loader = DataLoader(
        validation_dataset,
        batch_size=int(CONFIG["EVAL_BATCH_SIZE"]),
        shuffle=False,
        num_workers=int(CONFIG["NUM_WORKERS"]),
        pin_memory=bool(
            CONFIG["PIN_MEMORY"] and DEVICE.type == "cuda"
        ),
        drop_last=False,
    )

    history_log = {
        "epoch": [],
        "lr": [],
        "train_relative_l2": [],
        "train_physical_l2": [],
        "val_relative_l2": [],
        "val_physical_l2": [],
        "val_recursive_full_l2": [],
    }
    best_epoch = -1
    best_val_full = float("inf")
    start_epoch = 1
    elapsed_before = 0.0

    if latest_path.exists() and not CONFIG["FORCE_RETRAIN"]:
        latest = torch_load_safe(latest_path, map_location=DEVICE)
        if int(latest["n_train"]) != int(n_train):
            raise RuntimeError("Checkpoint sample-size mismatch.")
        if int(latest["history_length"]) != int(
            CONFIG["HISTORY_LENGTH"]
        ):
            raise RuntimeError("Checkpoint history-length mismatch.")
        model.load_state_dict(latest["model_state"])
        optimizer.load_state_dict(latest["optimizer_state"])
        optimizer_to_device(optimizer, DEVICE)
        scheduler.load_state_dict(latest["scheduler_state"])
        history_log = latest["history"]
        best_epoch = int(latest["best_epoch"])
        best_val_full = float(latest["best_val_full"])
        start_epoch = int(latest["epoch"]) + 1
        elapsed_before = float(latest.get("train_time_sec", 0.0))
        state = latest.get("shuffle_generator_state")
        if state is not None:
            if not torch.is_tensor(state):
                state = torch.as_tensor(state, dtype=torch.uint8)
            state = (
                state.detach()
                .cpu()
                .to(dtype=torch.uint8)
                .contiguous()
            )
            shuffle_generator.set_state(state)
        print(f"Resuming N={n_train} at epoch {start_epoch}.")

    print("\n" + "=" * 100)
    print(
        f"UNO2D-H{CONFIG['HISTORY_LENGTH']} | N_train={n_train}"
    )
    print("=" * 100)
    print("exact input states      :", CONFIG["HISTORY_LENGTH"])
    print("training rollout steps  :", CONFIG["TRAIN_ROLLOUT_STEPS"])
    print("training segments       :", len(train_dataset))
    print("validation segments     :", len(validation_dataset))
    print("parameters              :", f"{expected_parameters:,}")
    print("stored real scalars     :", f"{expected_real_scalars:,}")
    print("batch size              :", CONFIG["BATCH_SIZE"])
    print("mass enforcement        :", CONFIG["ENFORCE_MASS"])
    print("test used for selection : no")

    segment_start = time.perf_counter()
    for epoch in range(start_epoch, int(CONFIG["EPOCHS"]) + 1):
        train_rel, train_physical = train_one_epoch(
            model, train_loader, optimizer
        )
        val_rel, val_physical = evaluate_windows(
            model, validation_loader
        )
        run_rollout = (
            epoch % int(CONFIG["VAL_ROLLOUT_INTERVAL"]) == 0
            or epoch == int(CONFIG["EPOCHS"])
        )
        val_full = (
            validation_rollout_error(model)
            if run_rollout
            else float("nan")
        )

        current_lr = float(optimizer.param_groups[0]["lr"])
        history_log["epoch"].append(epoch)
        history_log["lr"].append(current_lr)
        history_log["train_relative_l2"].append(train_rel)
        history_log["train_physical_l2"].append(train_physical)
        history_log["val_relative_l2"].append(val_rel)
        history_log["val_physical_l2"].append(val_physical)
        history_log["val_recursive_full_l2"].append(val_full)

        if (
            run_rollout
            and np.isfinite(val_full)
            and val_full < best_val_full
        ):
            best_epoch = int(epoch)
            best_val_full = float(val_full)
            torch.save(
                {
                    "n_train": int(n_train),
                    "history_length": int(
                        CONFIG["HISTORY_LENGTH"]
                    ),
                    "model_state": state_dict_to_cpu(model),
                    "epoch": best_epoch,
                    "best_val_full": best_val_full,
                    "config": copy.deepcopy(CONFIG),
                },
                best_path,
            )

        scheduler.step()
        if epoch == 1 or epoch % 10 == 0:
            print(
                f"[UNO2D-H{CONFIG['HISTORY_LENGTH']}] "
                f"{epoch:04d}/{CONFIG['EPOCHS']} | "
                f"train rel {train_rel:.6e} | "
                f"val rel {val_rel:.6e} | "
                f"train physical {train_physical:.6e} | "
                f"val physical {val_physical:.6e} | "
                f"val recursive full {val_full:.6e} | "
                f"lr {current_lr:.3e}"
            )

        if (
            epoch % int(CONFIG["CHECKPOINT_INTERVAL"]) == 0
            or epoch == int(CONFIG["EPOCHS"])
        ):
            elapsed = elapsed_before + (
                time.perf_counter() - segment_start
            )
            torch.save(
                {
                    "n_train": int(n_train),
                    "history_length": int(
                        CONFIG["HISTORY_LENGTH"]
                    ),
                    "epoch": int(epoch),
                    "model_state": state_dict_to_cpu(model),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                    "history": history_log,
                    "best_epoch": int(best_epoch),
                    "best_val_full": float(best_val_full),
                    "train_time_sec": float(elapsed),
                    "shuffle_generator_state": (
                        shuffle_generator.get_state()
                    ),
                    "config": copy.deepcopy(CONFIG),
                },
                latest_path,
            )

    total_train_time = elapsed_before + (
        time.perf_counter() - segment_start
    )
    if not best_path.exists():
        raise RuntimeError(
            f"No finite validation checkpoint for N={n_train}."
        )
    best = torch_load_safe(best_path, map_location=DEVICE)
    model.load_state_dict(best["model_state"])

    done_bundle = {
        "n_train": int(n_train),
        "history_length": int(CONFIG["HISTORY_LENGTH"]),
        "model_state": state_dict_to_cpu(model),
        "history": history_log,
        "best_epoch": int(best["epoch"]),
        "best_val_full": float(best["best_val_full"]),
        "train_time_sec": float(total_train_time),
        "parameters": int(count_parameters(model)),
        "stored_real_scalars": int(count_real_scalars(model)),
        "config": copy.deepcopy(CONFIG),
    }
    torch.save(done_bundle, done_path)
    return {
        "model": model,
        "history": history_log,
        "best_epoch": int(best["epoch"]),
        "best_val_full": float(best["best_val_full"]),
        "train_time_sec": float(total_train_time),
        "checkpoint": str(done_path),
    }


def timed_test_rollout(model):
    initial_history = U_TEST[
        :, :, :, : CONFIG["HISTORY_LENGTH"]
    ]
    with torch.no_grad():
        warm_history = torch.as_tensor(
            initial_history[: min(2, len(initial_history))],
            dtype=torch.float32,
            device=DEVICE,
        )
        for _ in range(3):
            warm_prediction = model(warm_history)
            warm_prediction = project_mass(
                warm_prediction, warm_history[:, :, :, -1]
            )
            warm_history = torch.cat(
                [
                    warm_history[:, :, :, 1:],
                    warm_prediction.unsqueeze(-1),
                ],
                dim=-1,
            )
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    timings = []
    prediction = None
    for _ in range(int(CONFIG["INFERENCE_REPEATS"])):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        prediction = recursive_rollout(
            model, initial_history, NT
        )
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        timings.append(time.perf_counter() - start)
    return prediction, float(np.median(timings)), timings

## Diagnostics and complete sample-size sweep

In [9]:
def forecast_only_metrics(prediction, truth):
    start = int(CONFIG["HISTORY_LENGTH"]) - 1
    l2_t = l2_per_timestep_np(prediction, truth)[:, start:]
    weights = time_weights(l2_t.shape[1])[None, :]
    values = np.sqrt(
        np.sum(weights * l2_t * l2_t, axis=1) * DT + 1.0e-30
    )
    return {
        "forecast_full_mean": float(np.mean(values)),
        "forecast_full_std": float(np.std(values)),
        "forecast_full_median": float(np.median(values)),
    }


def plot_history(history, n_train):
    if not CONFIG["SAVE_PLOTS"]:
        return
    epochs = np.asarray(history["epoch"], dtype=int)
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].semilogy(
        epochs, history["train_relative_l2"], label="train"
    )
    axes[0].semilogy(
        epochs, history["val_relative_l2"], label="validation"
    )
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("mean relative L2 per unrolled step")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    val_full = np.asarray(
        history["val_recursive_full_l2"], dtype=float
    )
    finite = np.isfinite(val_full)
    axes[1].semilogy(
        epochs[finite],
        val_full[finite],
        marker="o",
        label="validation full-rollout physical L2",
    )
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("absolute trajectory L2")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    figure.suptitle(
        f"UNO2D-H{CONFIG['HISTORY_LENGTH']}, N={n_train}"
    )
    figure.tight_layout()
    path = PLOT_DIR / (
        f"N{n_train}_uno2d_h{CONFIG['HISTORY_LENGTH']}_training.png"
    )
    figure.savefig(path, dpi=160)
    plt.close(figure)


def plot_l2_time(metrics, n_train):
    if not CONFIG["SAVE_PLOTS"]:
        return
    figure, axis = plt.subplots(figsize=(7.5, 4.5))
    axis.plot(t_grid, metrics["mean_t"], label="mean", linewidth=2)
    axis.plot(
        t_grid, metrics["median_t"], "--", label="median", linewidth=2
    )
    axis.fill_between(
        t_grid,
        np.maximum(metrics["mean_t"] - metrics["std_t"], 0.0),
        metrics["mean_t"] + metrics["std_t"],
        alpha=0.18,
        label="mean +/- std",
    )
    axis.set_xlabel("time")
    axis.set_ylabel("absolute physical L2")
    axis.set_title(
        f"UNO2D-H{CONFIG['HISTORY_LENGTH']} rollout, N={n_train}"
    )
    axis.grid(True, alpha=0.3)
    axis.legend()
    figure.tight_layout()
    path = PLOT_DIR / (
        f"N{n_train}_uno2d_h{CONFIG['HISTORY_LENGTH']}_l2_time.png"
    )
    figure.savefig(path, dpi=160)
    plt.close(figure)


all_results = []
for n_train in CONFIG["RUN_SAMPLE_SIZES"]:
    if n_train not in CONFIG["SAMPLE_SIZES"]:
        raise ValueError(f"Unexpected sample size {n_train}.")

    fitted = fit_or_load_uno2d(n_train)
    prediction, inference_median, inference_all = timed_test_rollout(
        fitted["model"]
    )
    if not np.all(np.isfinite(prediction)):
        raise FloatingPointError(
            f"Non-finite test rollout for N={n_train}."
        )
    metrics = rollout_metrics(prediction, U_TEST)
    forecast = forecast_only_metrics(prediction, U_TEST)
    mass_drift = mass_drift_np(prediction)

    result = {
        "n_train": int(n_train),
        "model": f"UNO2D-H{CONFIG['HISTORY_LENGTH']}-W32-7BLOCK",
        "parameters": int(count_parameters(fitted["model"])),
        "stored_real_scalars": int(
            count_real_scalars(fitted["model"])
        ),
        "best_epoch": int(fitted["best_epoch"]),
        "validation_full_l2": float(fitted["best_val_full"]),
        "full_mean": metrics["full_mean"],
        "full_std": metrics["full_std"],
        "full_median": metrics["full_median"],
        "forecast_full_mean": forecast["forecast_full_mean"],
        "forecast_full_std": forecast["forecast_full_std"],
        "forecast_full_median": forecast["forecast_full_median"],
        "final_mean": metrics["final_mean"],
        "final_std": metrics["final_std"],
        "final_median": metrics["final_median"],
        "mean_mass_drift": float(np.mean(mass_drift)),
        "train_time_sec": float(fitted["train_time_sec"]),
        "inference_time_median_sec": inference_median,
        "inference_time_all_sec": inference_all,
        "checkpoint": fitted["checkpoint"],
    }
    all_results.append(result)

    tag = f"N{n_train}_uno2d_h{CONFIG['HISTORY_LENGTH']}"
    np.savez_compressed(
        RESULT_DIR / f"{tag}_metrics.npz",
        mean_t=metrics["mean_t"],
        median_t=metrics["median_t"],
        std_t=metrics["std_t"],
        all_full=metrics["all_full"],
        all_final=metrics["all_final"],
        mass_drift=mass_drift,
        inference_times=np.asarray(inference_all),
    )
    if CONFIG["SAVE_ROLLOUTS"]:
        np.save(
            RESULT_DIR / f"{tag}_rollout.npy",
            prediction.astype(np.float32),
        )

    plot_history(fitted["history"], n_train)
    plot_l2_time(metrics, n_train)

    print("\nTest result")
    print("-----------")
    print(f"model             : {result['model']}")
    print(f"N_train           : {n_train}")
    print(f"parameters        : {result['parameters']:,}")
    print(f"real scalars      : {result['stored_real_scalars']:,}")
    print(
        f"full mean/std     : {result['full_mean']:.6e} / "
        f"{result['full_std']:.6e}"
    )
    print(
        f"forecast-only L2  : {result['forecast_full_mean']:.6e}"
    )
    print(f"full median       : {result['full_median']:.6e}")
    print(
        f"final mean/std    : {result['final_mean']:.6e} / "
        f"{result['final_std']:.6e}"
    )
    print(f"final median      : {result['final_median']:.6e}")
    print(f"mean mass drift   : {result['mean_mass_drift']:.6e}")
    print(f"training time     : {result['train_time_sec']:.2f} s")
    print(
        f"inference median  : "
        f"{result['inference_time_median_sec']:.4f} s"
    )

    del fitted, prediction, metrics, forecast, mass_drift
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


summary = pd.DataFrame(all_results)
display(summary)
history_label = f"h{CONFIG['HISTORY_LENGTH']}"
csv_path = RESULT_DIR / f"cahn_hilliard_uno2d_{history_label}_summary.csv"
json_path = RESULT_DIR / f"cahn_hilliard_uno2d_{history_label}_summary.json"
summary.to_csv(csv_path, index=False)
with open(json_path, "w") as stream:
    json.dump(json_ready(all_results), stream, indent=2)

print("\n" + "=" * 144)
print(
    f"CAHN--HILLIARD UNO2D-H{CONFIG['HISTORY_LENGTH']}: "
    "ABSOLUTE PHYSICAL L2"
)
print("=" * 144)
for row in all_results:
    print(
        f"N={row['n_train']:4d} | "
        f"full mean {row['full_mean']:.6e} | "
        f"std {row['full_std']:.6e} | "
        f"median {row['full_median']:.6e} | "
        f"forecast-only {row['forecast_full_mean']:.6e} | "
        f"final mean {row['final_mean']:.6e} | "
        f"final median {row['final_median']:.6e} | "
        f"params {row['parameters']:,} | "
        f"real scalars {row['stored_real_scalars']:,} | "
        f"train {row['train_time_sec']:.2f}s | "
        f"inference {row['inference_time_median_sec']:.4f}s"
    )
print("=" * 144)
print("Saved CSV :", csv_path)
print("Saved JSON:", json_path)


UNO2D-H10 | N_train=25
exact input states      : 10
training rollout steps  : 40
training segments       : 75
validation segments     : 90
parameters              : 3,899,089
stored real scalars     : 7,696,081
batch size              : 16
mass enforcement        : True
test used for selection : no
[UNO2D-H10] 0001/500 | train rel 1.000415e+00 | val rel 9.978626e-01 | train physical 1.386348e+00 | val physical 1.387240e+00 | val recursive full nan | lr 1.000e-03
[UNO2D-H10] 0010/500 | train rel 6.538881e-01 | val rel 6.580100e-01 | train physical 8.458706e-01 | val physical 8.551458e-01 | val recursive full 1.386594e+00 | lr 1.000e-03
[UNO2D-H10] 0020/500 | train rel 3.046859e-01 | val rel 3.877967e-01 | train physical 3.790485e-01 | val physical 5.250264e-01 | val recursive full 1.048450e+00 | lr 1.000e-03
[UNO2D-H10] 0030/500 | train rel 1.605223e-01 | val rel 3.138704e-01 | train physical 1.932917e-01 | val physical 4.399233e-01 | val recursive full 8.269661e-01 | lr 1.000e-03
[UNO

,n_train,model,parameters,stored_real_scalars,best_epoch,validation_full_l2,full_mean,full_std,full_median,forecast_full_mean,forecast_full_std,forecast_full_median,final_mean,final_std,final_median,mean_mass_drift,train_time_sec,inference_time_median_sec,inference_time_all_sec,checkpoint
0,25,UNO2D-H10-W32-7BLOCK,3899089,7696081,170,0.715006,0.707293,0.087361,0.693165,0.707293,0.087361,0.693165,1.187028,0.166763,1.170288,0.000119,637.397620,0.573504,"[0.5783556350002073, 0.5717929939996793, 0.573...",/content/gdrive/MyDrive/anus/cahn_hilliard_uno...
1,50,UNO2D-H10-W32-7BLOCK,3899089,7696081,400,0.623100,0.600874,0.091666,0.583763,0.600874,0.091666,0.583763,1.019405,0.176320,0.986559,0.000119,1097.626760,0.577944,"[0.5815806370001155, 0.5779444360000525, 0.577...",/content/gdrive/MyDrive/anus/cahn_hilliard_uno...
2,100,UNO2D-H10-W32-7BLOCK,3899089,7696081,270,0.502990,0.487296,0.086975,0.473823,0.487296,0.086975,0.473823,0.841558,0.184704,0.813609,0.000119,1886.535602,0.579737,"[0.5813716960001329, 0.5792195760004688, 0.579...",/content/gdrive/MyDrive/anus/cahn_hilliard_uno...
3,200,UNO2D-H10-W32-7BLOCK,3899089,7696081,500,0.378011,0.383282,0.071049,0.380248,0.383282,0.071049,0.380248,0.684007,0.150548,0.667761,0.000119,3626.532249,0.576557,"[0.579123724000965, 0.5762249330000486, 0.5765...",/content/gdrive/MyDrive/anus/cahn_hilliard_uno...
4,400,UNO2D-H10-W32-7BLOCK,3899089,7696081,350,0.261645,0.257588,0.062749,0.243568,0.257588,0.062749,0.243568,0.469261,0.149074,0.466918,0.000119,6976.552812,0.574856,"[0.5746159519985667, 0.5764432940013648, 0.574...",/content/gdrive/MyDrive/anus/cahn_hilliard_uno...



CAHN--HILLIARD UNO2D-H10: ABSOLUTE PHYSICAL L2
N=  25 | full mean 7.072926e-01 | std 8.736076e-02 | median 6.931653e-01 | forecast-only 7.072926e-01 | final mean 1.187028e+00 | final median 1.170288e+00 | params 3,899,089 | real scalars 7,696,081 | train 637.40s | inference 0.5735s
N=  50 | full mean 6.008739e-01 | std 9.166570e-02 | median 5.837628e-01 | forecast-only 6.008739e-01 | final mean 1.019405e+00 | final median 9.865585e-01 | params 3,899,089 | real scalars 7,696,081 | train 1097.63s | inference 0.5779s
N= 100 | full mean 4.872964e-01 | std 8.697496e-02 | median 4.738234e-01 | forecast-only 4.872964e-01 | final mean 8.415578e-01 | final median 8.136091e-01 | params 3,899,089 | real scalars 7,696,081 | train 1886.54s | inference 0.5797s
N= 200 | full mean 3.832823e-01 | std 7.104950e-02 | median 3.802480e-01 | forecast-only 3.832823e-01 | final mean 6.840074e-01 | final median 6.677611e-01 | params 3,899,089 | real scalars 7,696,081 | train 3626.53s | inference 0.5766s
N= 40

## Reporting labels

- H1: `UNO2D-H1 (one-state autoregressive, Cahn--Hilliard adaptation)`.
- H10: `UNO2D-H10 (official temporal protocol, Cahn--Hilliard adaptation)`.
- H10 receives ten exact test states and must not be described as
  input-matched to AWFNO.
- Both variants output one state per call and are not windowed-output models.
- Parameter entries count complex tensors once per complex number; the
  `stored_real_scalars` column counts each complex number as two real values.

In [10]:
from google.colab import runtime
runtime.unassign()